In [1]:
import os
import weaviate
from dotenv import load_dotenv
from weaviate.classes.init import Auth

In [2]:
load_dotenv()

True

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.getenv("WEAVIATE_URL"),
    auth_credentials=Auth.api_key(os.getenv("WEAVIATE_API_KEY")),
)

print(client.is_ready())

True


In [4]:
# Load the local all-MiniLM-L6-v2 model (downloaded by download_embedding_models.py)
from pathlib import Path
from sentence_transformers import SentenceTransformer

MODEL_DIR = Path.cwd() / "embedding_model" / "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(str(MODEL_DIR), device="cpu")

def embed_texts(texts):
    """Embed a list of strings locally -> list of float vectors (384-dim)."""
    return embedding_model.encode(texts, normalize_embeddings=True).tolist()

def embed_query(query):
    """Embed a single query string locally."""
    return embedding_model.encode(query, normalize_embeddings=True).tolist()

print("Embedding dimension:", embedding_model.get_embedding_dimension())

d:\AI Engineering\RAG-LAB\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4429.00it/s]

Embedding dimension: 384


In [5]:
# Create the Weaviate schema (collection) with NO vectorizer module.
# Vectors are "self provided": we compute them with the local MiniLM model,
# so no OpenAI / Cohere / HF API key is needed.
from weaviate.classes.config import Configure, Property, DataType, VectorDistances

COLLECTION_NAME = "AgenticDesignPatterns"
RECREATE = False  # set True to drop and rebuild the collection

if RECREATE and client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

if not client.collections.exists(COLLECTION_NAME):
    client.collections.create(
        name=COLLECTION_NAME,
        description="Chunks of the Agentic Design Patterns PDF for hybrid search",
        vector_config=Configure.Vectors.self_provided(
            # This Weaviate Cloud cluster only allows the HFresh index type (not HNSW)
            vector_index_config=Configure.VectorIndex.hfresh(
                distance_metric=VectorDistances.COSINE
            ),
        ),
        # BM25 keyword index on text props powers the sparse side of hybrid search
        inverted_index_config=Configure.inverted_index(bm25_b=0.75, bm25_k1=1.2),
        properties=[
            Property(name="content", data_type=DataType.TEXT),
            Property(name="source", data_type=DataType.TEXT, skip_vectorization=True),
            Property(name="page", data_type=DataType.INT),
            Property(name="chunk_id", data_type=DataType.INT),
        ],
    )
    print(f"Collection '{COLLECTION_NAME}' created.")
else:
    print(f"Collection '{COLLECTION_NAME}' already exists.")

collection = client.collections.use(COLLECTION_NAME)
print(collection.config.get().vector_config)

Collection 'AgenticDesignPatterns' already exists.
{'default': _NamedVectorConfig(vectorizer=_NamedVectorizerConfig(vectorizer=<Vectorizers.NONE: 'none'>, model={}, source_properties=None), vector_index_config=_VectorIndexConfigHFresh(multi_vector=None, quantizer=_RQConfig(cache=None, bits=1, rescore_limit=350, centering=False, training_limit=0), distance_metric=<VectorDistances.COSINE: 'cosine'>, max_posting_size_kb=48, replicas=4, search_probe=256))}


In [6]:
# View the schema. In weaviate-client v4 `client.schema` was removed:
# a schema "class" is now a "collection", reached via `client.collections`.
for name, cfg in client.collections.list_all(simple=False).items():
    print(name, [(p.name, p.data_type.value) for p in cfg.properties])

# Full schema of our collection as a dict
collection.config.get().to_dict()

AgenticDesignPatterns [('content', 'text'), ('source', 'text'), ('page', 'int'), ('chunk_id', 'int')]


{'description': 'Chunks of the Agentic Design Patterns PDF for hybrid search',
 'invertedIndexConfig': {'bm25': {'b': 0.75, 'k1': 1.2},
  'cleanupIntervalSeconds': 60,
  'indexNullState': False,
  'indexPropertyLength': False,
  'indexTimestamps': False,
  'stopwords': {'preset': 'en'}},
 'multiTenancyConfig': {'enabled': False,
  'autoTenantCreation': False,
  'autoTenantActivation': False},
 'properties': [{'name': 'content',
   'dataType': ['text'],
   'indexFilterable': True,
   'indexSearchable': True,
   'indexRangeFilters': False,
   'tokenization': 'word'},
  {'name': 'source',
   'dataType': ['text'],
   'indexFilterable': True,
   'indexSearchable': True,
   'indexRangeFilters': False,
   'tokenization': 'word'},
  {'name': 'page',
   'dataType': ['int'],
   'indexFilterable': True,
   'indexSearchable': False,
   'indexRangeFilters': False,
   'tokenization': None},
  {'name': 'chunk_id',
   'dataType': ['int'],
   'indexFilterable': True,
   'indexSearchable': False,
   'in

In [7]:
# Custom LangChain retriever for the Weaviate v4 client.
# LangChain's WeaviateHybridSearchRetriever (langchain_community) only works with the old
# v3 client (client.schema / client.query), so we rebuild it with the same attributes.
#
# How it works:
#   * Typed class attributes are Pydantic fields -> they become constructor arguments
#     (client=..., index_name=..., alpha=...). No __init__ needed.
#   * The @model_validator runs once when the object is created. Like the original class,
#     it handles `create_schema_if_missing` (creates the collection if it doesn't exist).
#   * `_get_relevant_documents` is the one method BaseRetriever requires us to override;
#     retriever.invoke(query) calls it and it runs the v4 hybrid (BM25 + vector) search.
from typing import Any, Callable, Dict, List, Optional
from uuid import uuid4

import weaviate
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from pydantic import ConfigDict, PrivateAttr, model_validator
from weaviate.classes.config import Configure, DataType, Property, VectorDistances
from weaviate.classes.query import MetadataQuery


class WeaviateV4HybridRetriever(BaseRetriever):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    client: Any                                  # weaviate.WeaviateClient (v4)
    index_name: str                              # collection name ("class" in v3)
    text_key: str = "content"                    # property holding the chunk text
    k: int = 4                                   # number of results
    alpha: float = 0.5                           # 0 = pure BM25 (keyword), 1 = pure vector
    attributes: List[str] = []                   # extra properties returned as metadata
    create_schema_if_missing: bool = True
    embed_fn: Optional[Callable[[str], List[float]]] = None  # query -> vector (defaults to embed_query)

    _collection: Any = PrivateAttr(default=None)

    @model_validator(mode="after")
    def _setup(self):
        if not isinstance(self.client, weaviate.WeaviateClient):
            raise ValueError(
                f"client should be a weaviate v4 WeaviateClient, got {type(self.client)}"
            )
        if self.create_schema_if_missing and not self.client.collections.exists(self.index_name):
            self.client.collections.create(
                name=self.index_name,
                # Vectors are computed locally (no vectorizer module). This cluster only allows HFresh.
                vector_config=Configure.Vectors.self_provided(
                    vector_index_config=Configure.VectorIndex.hfresh(
                        distance_metric=VectorDistances.COSINE
                    ),
                ),
                properties=[Property(name=self.text_key, data_type=DataType.TEXT)]
                + [Property(name=a, data_type=DataType.TEXT) for a in self.attributes],
            )
        self._collection = self.client.collections.use(self.index_name)
        if self.embed_fn is None:
            self.embed_fn = embed_query
        return self

    def add_documents(self, docs: List[Document]) -> List[str]:
        """Embed docs locally and upload them (text -> text_key, metadata -> other properties)."""
        vectors = embed_texts([d.page_content for d in docs])
        ids = []
        with self._collection.batch.dynamic() as batch:
            for doc, vec in zip(docs, vectors):
                uid = str(uuid4())
                batch.add_object(
                    properties={self.text_key: doc.page_content, **doc.metadata},
                    vector=vec,
                    uuid=uid,
                )
                ids.append(uid)
        failed = self._collection.batch.failed_objects
        if failed:
            raise RuntimeError(f"{len(failed)} objects failed to upload: {failed[0].message}")
        return ids

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> List[Document]:
        res = self._collection.query.hybrid(
            query=query,
            vector=self.embed_fn(query),  # no vectorizer on the collection, so pass our own vector
            alpha=self.alpha,
            limit=self.k,
            return_properties=[self.text_key, *self.attributes] if self.attributes else None,
            return_metadata=MetadataQuery(score=True),
        )
        docs = []
        for o in res.objects:
            props: Dict[str, Any] = dict(o.properties)
            text = props.pop(self.text_key, "")
            docs.append(Document(page_content=text, metadata={**props, "score": o.metadata.score}))
        return docs


retriever = WeaviateV4HybridRetriever(
    client=client,
    index_name=COLLECTION_NAME,
    text_key="content",
    attributes=["source", "page", "chunk_id"],
    alpha=0.5,
    k=4,
    create_schema_if_missing=True,
)

In [8]:
# Load the PDF, split it into chunks and upload them to Weaviate.
# Each chunk is stored as: content (text) + source, page, chunk_id (metadata) + its MiniLM vector.
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import re

PDF_PATH = Path.cwd() / "data" / "Agentic_Design_Patterns.pdf"

pages = PyPDFLoader(str(PDF_PATH)).load()

# This PDF extracts with line breaks between almost every word.
# Collapse all whitespace to single spaces so BM25 and the embeddings see normal sentences.
for p in pages:
    p.page_content = re.sub(r"\s+", " ", p.page_content).strip()
pages = [p for p in pages if p.page_content]  # drop blank pages

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(pages)

# Keep only the metadata that matches our collection's properties
chunks = [
    Document(
        page_content=c.page_content,
        metadata={"source": PDF_PATH.name, "page": int(c.metadata.get("page", 0)), "chunk_id": i},
    )
    for i, c in enumerate(chunks)
]
print(f"{len(pages)} pages -> {len(chunks)} chunks")

# Only upload if the collection is empty (so re-running this cell doesn't create duplicates)
existing = collection.aggregate.over_all(total_count=True).total_count
if existing == 0:
    ids = retriever.add_documents(chunks)
    print(f"Uploaded {len(ids)} chunks.")
else:
    print(f"Collection already has {existing} objects - skipping upload.")

print("Objects in collection:", collection.aggregate.over_all(total_count=True).total_count)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_18232\1064880447.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


482 pages -> 1135 chunks
Collection already has 1135 objects - skipping upload.
Objects in collection: 1135


In [9]:
# Test the hybrid retriever (alpha: 0 = keyword/BM25 only, 1 = vector only)
query = "What is the reflection pattern?"
for i, doc in enumerate(retriever.invoke(query), 1):
    print(f"--- Result {i} | page {doc.metadata['page']} | score {doc.metadata['score']:.3f}")
    print(doc.page_content[:300], "\n")

--- Result 1 | page 74 | score 0.921
Fig. 1: Reflection design pattern, self-reflection 11 

--- Result 2 | page 477 | score 0.829
the "worker" agent. This critic can be given specific criteria to look for, leading to more rigorous and targeted improvements. How does reflection help in reducing "hallucinations"? By prompting an agent to review its work, especially by comparing its statements against a known source or by checkin 

--- Result 3 | page 477 | score 0.750
How does the "Reflection" pattern improve an agent's performance? Reflection acts as a form of self-correction. After generating a response or completing a task, the agent can be prompted to review its work, check for errors, assess its quality against certain criteria, or consider alternative appro 

--- Result 4 | page 477 | score 0.737
enough for high-quality output? Simple prompt chaining (where the output of one prompt becomes the input for the next) is often too basic. The model might just rephrase its previous outpu

In [10]:
# Synthesis: Gemini LLM + RetrievalQA chain on top of the Weaviate hybrid retriever
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

llm = ChatGoogleGenerativeAI(
    model="gemini-3.7-flash",
    temperature=0,
)

# RetrievalQA's "stuff" chain fills {context} with the retrieved chunks and {question} with the query
qa_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a helpful AI assistant.

Answer the user's question using ONLY the provided context.

If the answer is not available in the context,
clearly state that you don't know.

Context:
{context}

Question: {question}

Answer:""",
)

hybrid_qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,  # so we can print the context retrieved from Weaviate
    chain_type_kwargs={"prompt": qa_prompt},
)

In [11]:
def ask(query: str):
    """Run the hybrid RetrievalQA chain and print the Weaviate context + Gemini's answer."""
    response = hybrid_qa_chain.invoke({"query": query})

    print("QUERY:", query)
    print("RETRIEVED CONTEXT FROM WEAVIATE:")
    for i, doc in enumerate(response["source_documents"], 1):
        print(f"Document {i}")
        print("Source:", doc.metadata.get("source"))
        print("Page:", doc.metadata.get("page"))
        print("Chunk ID:", doc.metadata.get("chunk_id"))
        print(f"Hybrid score: {doc.metadata.get('score', 0):.3f}")
        print("Content:", doc.page_content)
        print("-" * 60)

    print("GENERATED ANSWER:")
    print(response["result"])
    return response

In [12]:
response1 = ask("What is the reflection pattern?")

QUERY: What is the reflection pattern?
RETRIEVED CONTEXT FROM WEAVIATE:
Document 1
Source: Agentic_Design_Patterns.pdf
Page: 74
Chunk ID: 185
Hybrid score: 0.921
Content: Fig. 1: Reflection design pattern, self-reflection 11
------------------------------------------------------------
Document 2
Source: Agentic_Design_Patterns.pdf
Page: 477
Chunk ID: 1120
Hybrid score: 0.829
Content: the "worker" agent. This critic can be given specific criteria to look for, leading to more rigorous and targeted improvements. How does reflection help in reducing "hallucinations"? By prompting an agent to review its work, especially by comparing its statements against a known source or by checking its own reasoning steps, the Reflection pattern can significantly reduce the likelihood of hallucinations (making up facts). The agent is forced to be more grounded in the provided context and less likely to generate unsupported information. Can the Reflection pattern be applied more than once? Yes, reflection

In [13]:
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

In [15]:
# langchain_cohere's CohereRerank has no default model, so `model` must be passed explicitly
compresor = CohereRerank(
    cohere_api_key=os.getenv("COHERE_API_KEY"),
    model="rerank-v3.5",
    top_n=3,
)

In [16]:
compression_retrieval = ContextualCompressionRetriever(
    base_compressor=compresor, base_retriever=retriever
)

In [17]:
# Use the public .invoke(); _get_relevant_documents is internal and needs a run_manager
compressed_docs = compression_retrieval.invoke("What is the reflection pattern?")
compressed_docs

[Document(metadata={'page': 477, 'source': 'Agentic_Design_Patterns.pdf', 'chunk_id': 1118, 'score': 0.7502793073654175, 'relevance_score': 0.8793233}, page_content='How does the "Reflection" pattern improve an agent\'s performance? Reflection acts as a form of self-correction. After generating a response or completing a task, the agent can be prompted to review its work, check for errors, assess its quality against certain criteria, or consider alternative approaches. This iterative refinement process helps the agent produce more accurate, relevant, and high-quality results. What is the core idea of the Reflection pattern? The Reflection pattern gives an agent the ability to step back and critique its own work. Instead of producing a final output in one go, the agent generates a draft and then "reflects" on it, identifying flaws, missing information, or areas for improvement. This self-correction process is key to enhancing the quality and accuracy of its responses. Why is simple "pro

In [18]:
hybrid_qa_chain_with_reranking = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=compression_retrieval,
    return_source_documents=True,  # so we can print the context retrieved from Weaviate
    chain_type_kwargs={"prompt": qa_prompt},
)

In [20]:
def ask(query: str):
    """Run the hybrid RetrievalQA chain and print the Weaviate context + Gemini's answer."""
    response = hybrid_qa_chain_with_reranking.invoke({"query": query})

    print("QUERY:", query)
    print("RETRIEVED CONTEXT FROM WEAVIATE and also reranked using COHERE:")
    for i, doc in enumerate(response["source_documents"], 1):
        print(f"Document {i}")
        print("Source:", doc.metadata.get("source"))
        print("Page:", doc.metadata.get("page"))
        print("Chunk ID:", doc.metadata.get("chunk_id"))
        print(f"Hybrid score: {doc.metadata.get('score', 0):.3f}")
        print("Content:", doc.page_content)
        print("-" * 60)

    print("GENERATED ANSWER:")
    print(response["result"])
    return response

In [21]:
response1 = ask("What is the reflection pattern?")

QUERY: What is the reflection pattern?
RETRIEVED CONTEXT FROM WEAVIATE and also reranked using COHERE:
Document 1
Source: Agentic_Design_Patterns.pdf
Page: 477
Chunk ID: 1118
Hybrid score: 0.750
Content: How does the "Reflection" pattern improve an agent's performance? Reflection acts as a form of self-correction. After generating a response or completing a task, the agent can be prompted to review its work, check for errors, assess its quality against certain criteria, or consider alternative approaches. This iterative refinement process helps the agent produce more accurate, relevant, and high-quality results. What is the core idea of the Reflection pattern? The Reflection pattern gives an agent the ability to step back and critique its own work. Instead of producing a final output in one go, the agent generates a draft and then "reflects" on it, identifying flaws, missing information, or areas for improvement. This self-correction process is key to enhancing the quality and accuracy